Cell 1

In [ ]:
import requests
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

OPENFDA_BASE = "https://api.fda.gov/drug/ndc.json"
DAILYMED_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"

#Cell 2

In [ ]:
class OpenFDAClient:

    def search_by_brand(self, drug_name):

        try:

            response = requests.get(
                OPENFDA_BASE,
                params={
                    "search": f'brand_name:"{drug_name}"',
                    "limit": 100
                },
                timeout=30
            )

            response.raise_for_status()

            return response.json().get(
                "results",
                []
            )

        except Exception as e:

            print(
                f"Brand Search Error: {e}"
            )

            return []

    def search_by_generic(self, drug_name):

        try:

            response = requests.get(
                OPENFDA_BASE,
                params={
                    "search": f'generic_name:"{drug_name}"',
                    "limit": 100
                },
                timeout=30
            )

            response.raise_for_status()

            return response.json().get(
                "results",
                []
            )

        except Exception as e:

            print(
                f"Generic Search Error: {e}"
            )

            return []

    def search_by_ndc(self, ndc):

        searches = [

            f'package_ndc:"{ndc}"',

            f'product_ndc:"{ndc}"'
        ]

        for search_string in searches:

            try:

                response = requests.get(
                    OPENFDA_BASE,
                    params={
                        "search": search_string,
                        "limit": 10
                    },
                    timeout=30
                )

                response.raise_for_status()

                results = response.json().get(
                    "results",
                    []
                )

                if results:

                    return results

            except:
                pass

        return []

    def build_products_df(
        self,
        records
    ):

        rows = []

        for record in records:

            rows.append({

                "product_ndc":
                    record.get(
                        "product_ndc"
                    ),

                "brand_name":
                    record.get(
                        "brand_name"
                    ),

                "generic_name":
                    record.get(
                        "generic_name"
                    ),

                "manufacturer":
                    record.get(
                        "labeler_name"
                    ),

                "dosage_form":
                    record.get(
                        "dosage_form"
                    ),

                "route":
                    ", ".join(
                        record.get(
                            "route",
                            []
                        )
                    ),

                "marketing_category":
                    record.get(
                        "marketing_category"
                    ),

                "product_type":
                    record.get(
                        "product_type"
                    ),

                "application_number":
                    record.get(
                        "application_number"
                    ),

                "dea_schedule":
                    record.get(
                        "dea_schedule"
                    ),

                "start_marketing_date":
                    record.get(
                        "start_marketing_date"
                    ),

                "end_marketing_date":
                    record.get(
                        "end_marketing_date"
                    )
            })

        return (
            pd.DataFrame(rows)
            .drop_duplicates()
        )

    def build_packages_df(
        self,
        records
    ):

        rows = []

        for record in records:

            product_ndc = record.get(
                "product_ndc"
            )

            for package in record.get(
                "packaging",
                []
            ):

                rows.append({

                    "product_ndc":
                        product_ndc,

                    "package_ndc":
                        package.get(
                            "package_ndc"
                        ),

                    "package_description":
                        package.get(
                            "description"
                        ),

                    "sample":
                        package.get(
                            "sample"
                        )
                })

        return (
            pd.DataFrame(rows)
            .drop_duplicates()
        )

#Cell 3

In [ ]:
class DailyMedClient:

    def get_ndc_record(
        self,
        ndc
    ):

        try:

            response = requests.get(
                f"{DAILYMED_BASE}/ndcs/{ndc}.json",
                timeout=30
            )

            if response.status_code != 200:

                return {}

            data = response.json().get(
                "data",
                []
            )

            if len(data) == 0:

                return {}

            return data[0]

        except:

            return {}

    def build_dailymed_df(
        self,
        ndc_list
    ):

        rows = []

        for ndc in ndc_list:

            record = self.get_ndc_record(
                ndc
            )

            if record:

                rows.append({

                    "package_ndc":
                        ndc,

                    "setid":
                        record.get(
                            "setid"
                        ),

                    "title":
                        record.get(
                            "title"
                        ),

                    "published_date":
                        record.get(
                            "published_date"
                        )
                })

        return pd.DataFrame(rows)

Cell 4

In [ ]:
class DrugMaster:

    def __init__(self):

        self.fda = OpenFDAClient()

        self.dailymed = DailyMedClient()

    def search_drug(
        self,
        drug_name
    ):

        print(
            f"Searching for {drug_name}"
        )

        records = (
            self.fda.search_by_brand(
                drug_name
            )
        )

        if len(records) == 0:

            records = (
                self.fda.search_by_generic(
                    drug_name
                )
            )

        products_df = (
            self.fda.build_products_df(
                records
            )
        )

        packages_df = (
            self.fda.build_packages_df(
                records
            )
        )

        dailymed_df = pd.DataFrame()

        if not packages_df.empty:

            ndcs = (

                packages_df[
                    "package_ndc"
                ]

                .dropna()

                .unique()

                .tolist()
            )

            dailymed_df = (
                self.dailymed
                .build_dailymed_df(
                    ndcs
                )
            )

        return {

            "products":
                products_df,

            "packages":
                packages_df,

            "dailymed":
                dailymed_df
        }

    def search_ndc(
        self,
        ndc
    ):

        records = (
            self.fda.search_by_ndc(
                ndc
            )
        )

        products_df = (
            self.fda.build_products_df(
                records
            )
        )

        packages_df = (
            self.fda.build_packages_df(
                records
            )
        )

        dailymed_df = (
            self.dailymed
            .build_dailymed_df(
                [ndc]
            )
        )

        return {

            "products":
                products_df,

            "packages":
                packages_df,

            "dailymed":
                dailymed_df
        }

Cell 5

In [ ]:
def show_results(results):

    for name, df in results.items():

        print("\n")
        print("=" * 80)
        print(name.upper())
        print("=" * 80)

        print(
            f"Rows: {len(df)}"
        )

        display(df)

Cell 6

In [ ]:
def export_excel(
    results,
    filename
):

    with pd.ExcelWriter(
        filename,
        engine="openpyxl"
    ) as writer:

        for sheet, df in results.items():

            df.to_excel(
                writer,
                sheet_name=sheet[:31],
                index=False
            )

    print(
        f"Saved {filename}"
    )

Cell 7

In [ ]:
drug_name = "Opdivo"

lookup = DrugMaster()

results = lookup.search_drug(
    drug_name
)

show_results(results)


Cell 8 - Call this cell only if you want a specific NDC lookup

In [ ]:
ndc_results = lookup.search_ndc(
    "0003-3756"
)

show_results(ndc_results)

Cell 9

In [ ]:
import os

def export_excel(
    results,
    drug_name
):

    filename = (
        f"DrugMaster_{drug_name}.xlsx"
        .replace(" ", "_")
        .replace("/", "_")
    )

    full_path = os.path.abspath(filename)

    with pd.ExcelWriter(
        filename,
        engine="openpyxl"
    ) as writer:

        for sheet, df in results.items():

            df.to_excel(
                writer,
                sheet_name=sheet[:31],
                index=False
            )

    print(f"Saved: {full_path}")

export_excel(
    results,
    drug_name
)

Cell 10

In [ ]:
class RxNormClient:

    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def get_rxcui(self, drug_name):

        try:

            response = requests.get(
                f"{self.BASE_URL}/rxcui.json",
                params={
                    "name": drug_name
                },
                timeout=30
            )

            response.raise_for_status()

            return (
                response.json()
                .get("idGroup", {})
                .get("rxnormId", [])
            )

        except Exception as e:

            print(
                f"RxNorm Error: {e}"
            )

            return []

    def get_properties(
        self,
        rxcui
    ):

        try:

            response = requests.get(
                f"{self.BASE_URL}/rxcui/{rxcui}/properties.json",
                timeout=30
            )

            response.raise_for_status()

            return (
                response.json()
                .get("properties", {})
            )

        except:

            return {}

    def get_related_concepts(
        self,
        rxcui
    ):

        ttys = [

            "IN",
            "MIN",
            "PIN",

            "SCD",
            "SBD",

            "SCDF",
            "SCDC",

            "GPCK",
            "BPCK"
        ]

        rows = []

        for tty in ttys:

            try:

                response = requests.get(
                    f"{self.BASE_URL}/rxcui/{rxcui}/related.json",
                    params={
                        "tty": tty
                    },
                    timeout=30
                )

                groups = (
                    response.json()
                    .get(
                        "relatedGroup",
                        {}
                    )
                    .get(
                        "conceptGroup",
                        []
                    )
                )

                for group in groups:

                    for concept in group.get(
                        "conceptProperties",
                        []
                    ):

                        rows.append({

                            "source_rxcui":
                                rxcui,

                            "related_rxcui":
                                concept.get(
                                    "rxcui"
                                ),

                            "name":
                                concept.get(
                                    "name"
                                ),

                            "tty":
                                tty
                        })

            except:
                pass

        return pd.DataFrame(rows)

Cell 11

In [ ]:
def rxnorm_lookup(
    drug_name
):

    rx = RxNormClient()

    rxcuis = rx.get_rxcui(
        drug_name
    )

    summary_rows = []

    concept_rows = []

    for rxcui in rxcuis:

        props = rx.get_properties(
            rxcui
        )

        summary_rows.append({

            "rxcui":
                rxcui,

            "name":
                props.get("name"),

            "tty":
                props.get("tty"),

            "synonym":
                props.get("synonym")
        })

        concept_df = (
            rx.get_related_concepts(
                rxcui
            )
        )

        if not concept_df.empty:

            concept_rows.extend(
                concept_df.to_dict(
                    "records"
                )
            )

    return {

        "rxnorm_summary":
            pd.DataFrame(
                summary_rows
            ),

        "rxnorm_concepts":
            pd.DataFrame(
                concept_rows
            )
    }

Cell 12

In [ ]:
rx_results = rxnorm_lookup(
    drug_name
)

for name, df in rx_results.items():

    print("\n")
    print(name)
    print("-" * 50)

    display(df.head(20))

Cell 13

In [ ]:
drug_results = lookup.search_drug(
    drug_name
)

rx_results = rxnorm_lookup(
    drug_name
)

combined_results = {

    **drug_results,

    **rx_results
}

Cell 14

In [ ]:
show_results(
    combined_results
)

Cell 15 - Ingredients

In [ ]:
def build_ingredients_df(records):

    rows = []

    for record in records:

        product_ndc = record.get(
            "product_ndc"
        )

        brand_name = record.get(
            "brand_name"
        )

        generic_name = record.get(
            "generic_name"
        )

        for ingredient in record.get(
            "active_ingredients",
            []
        ):

            rows.append({

                "product_ndc":
                    product_ndc,

                "brand_name":
                    brand_name,

                "generic_name":
                    generic_name,

                "ingredient_name":
                    ingredient.get(
                        "name"
                    ),

                "strength":
                    ingredient.get(
                        "strength"
                    )
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Cell 16 - Build Ingredients

In [ ]:
fda_records = lookup.fda.search_by_brand(
    drug_name
)

if not fda_records:

    fda_records = lookup.fda.search_by_generic(
        drug_name
    )

ingredients_df = build_ingredients_df(
    fda_records
)

display(
    ingredients_df
)

Cell 17 - Pharmacologic Class

In [ ]:
def build_pharm_class_df(records):

    rows = []

    for record in records:

        classes = record.get(
            "pharm_class",
            []
        )

        for pharm_class in classes:

            rows.append({

                "product_ndc":
                    record.get(
                        "product_ndc"
                    ),

                "brand_name":
                    record.get(
                        "brand_name"
                    ),

                "pharm_class":
                    pharm_class
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Cell 18 - Test Pharmacologic Class

In [ ]:
pharm_class_df = build_pharm_class_df(
    fda_records
)

display(
    pharm_class_df
)

Cell 19 - Product Master Table

In [ ]:
products_df = results["products"]
packages_df = results["packages"]

master_df = products_df.merge(

    packages_df,

    on="product_ndc",

    how="left"
)

if not ingredients_df.empty:

    master_df = master_df.merge(

        ingredients_df,

        on=[
            "product_ndc",
            "brand_name",
            "generic_name"
        ],

        how="left"
    )

display(
    master_df
)